Field Assignment 03: Regional Prioritization for Operational Investment

Business Question

Which regions (and the stores within them) should Glorystone prioritize for operational improvement investment over the next 90 days, based on a balanced view of volume, fulfillment performance, customer experience, and revenue exposure?
Leadership needs a clear, defensible answer to “Where should limited attention and resources produce the highest return?” — not another list of problems.

Setup

In [1]:
import pandas as pd

Load CSV

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/CSRodgers184/Glorystone-DataAnalytics/main/01-online-fulfillment-performance/data/glorystone_online_pickup_orders.csv")

Feature Engineering




In [5]:
# Parse datetimes
df['order_datetime'] = pd.to_datetime(df['order_datetime'])
df['promised_ready_datetime'] = pd.to_datetime(df['promised_ready_datetime'])
df['actual_ready_datetime'] = pd.to_datetime(df['actual_ready_datetime'])
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

# Core metrics
df['readiness_delay_min'] = (df['actual_ready_datetime'] - df['promised_ready_datetime']).dt.total_seconds() / 60
df['wait_min'] = (df['pickup_datetime'] - df['actual_ready_datetime']).dt.total_seconds() / 60
df['is_abandoned'] = df['pickup_datetime'].isna()
df['fill_rate'] = df['items_fulfilled'] / df['items_ordered'].replace(0, pd.NA)
df['on_time_ready'] = df['readiness_delay_min'] <= 0

Regional Aggregation

In [6]:

#What this shows
#Scale (orders + revenue)
#Fulfillment performance (delay, on-time, fill, complete)
#Customer experience (wait + abandonment)

regional_summary = (
    df.groupby('region')
    .agg(
        orders=('order_id', 'count'),
        total_revenue=('order_value', 'sum'),
        avg_order_value=('order_value', 'mean'),
        avg_readiness_delay=('readiness_delay_min', 'mean'),
        pct_on_time_ready=('on_time_ready', 'mean'),
        avg_fill_rate=('fill_rate', 'mean'),
        pct_complete=('is_complete', 'mean'),
        avg_wait=('wait_min', 'mean'),
        abandonment_rate=('is_abandoned', 'mean')
    )
    .round(3)
    .sort_values('orders', ascending=False)
)

# Format for readability
regional_summary['total_revenue'] = regional_summary['total_revenue'].round(0).astype(int)
regional_summary['avg_order_value'] = regional_summary['avg_order_value'].round(2)
regional_summary['avg_readiness_delay'] = regional_summary['avg_readiness_delay'].round(1)
regional_summary['pct_on_time_ready'] = (regional_summary['pct_on_time_ready'] * 100).round(1)
regional_summary['avg_fill_rate'] = (regional_summary['avg_fill_rate'] * 100).round(1)
regional_summary['pct_complete'] = (regional_summary['pct_complete'] * 100).round(1)
regional_summary['avg_wait'] = regional_summary['avg_wait'].round(1)
regional_summary['abandonment_rate'] = (regional_summary['abandonment_rate'] * 100).round(1)

regional_summary

,orders,total_revenue,avg_order_value,avg_readiness_delay,pct_on_time_ready,avg_fill_rate,pct_complete,avg_wait,abandonment_rate
region,,,,,,,,,
North,2115,272844,129.00,6.1,25.3,94.62334,47.5,36.0,7.2
Central,2110,283351,134.29,12.7,18.0,90.421657,25.3,35.2,7.5
South,2103,276149,131.31,13.2,15.1,90.762335,29.1,36.2,6.8
East,1487,190951,128.41,9.6,18.0,92.021382,33.4,35.9,6.7
West,685,88475,129.16,8.2,21.3,92.899854,35.9,35.8,5.5


Store level Prioritization

In [8]:
store_summary = (
    df.groupby(['region', 'store_id'])
    .agg(
        orders=('order_id', 'count'),
        total_revenue=('order_value', 'sum'),
        avg_readiness_delay=('readiness_delay_min', 'mean'),
        pct_on_time_ready=('on_time_ready', 'mean'),
        avg_fill_rate=('fill_rate', 'mean'),
        pct_complete=('is_complete', 'mean'),
        avg_wait=('wait_min', 'mean'),
        abandonment_rate=('is_abandoned', 'mean')
    )
    .round(3)
    .sort_values('avg_readiness_delay', ascending=False)
)

# Format for readability
store_summary['total_revenue'] = store_summary['total_revenue'].round(0).astype(int)
store_summary['avg_readiness_delay'] = store_summary['avg_readiness_delay'].round(1)
store_summary['pct_on_time_ready'] = (store_summary['pct_on_time_ready'] * 100).round(1)
store_summary['avg_fill_rate'] = (store_summary['avg_fill_rate'] * 100).round(1)
store_summary['pct_complete'] = (store_summary['pct_complete'] * 100).round(1)
store_summary['avg_wait'] = store_summary['avg_wait'].round(1)
store_summary['abandonment_rate'] = (store_summary['abandonment_rate'] * 100).round(1)

store_summary

,,orders,total_revenue,avg_readiness_delay,pct_on_time_ready,avg_fill_rate,pct_complete,avg_wait,abandonment_rate
region,store_id,,,,,,,,
South,GS-09,662,84397,25.2,6.0,84.633838,7.6,36.3,7.7
Central,GS-04,655,83945,19.4,10.2,86.321593,10.7,35.4,9.5
East,GS-11,731,96870,12.3,15.9,90.109511,22.8,35.7,7.8
Central,GS-06,718,98640,10.6,18.4,91.169464,27.6,35.6,6.0
South,GS-08,706,97556,9.3,18.8,92.142587,29.2,38.0,6.8
Central,GS-05,725,99311,8.6,24.6,93.418048,36.1,34.6,7.3
West,GS-12,681,87803,8.2,21.3,92.926716,36.1,35.7,5.4
North,GS-02,713,90806,7.4,22.3,93.775902,39.4,37.1,7.6
East,GS-10,749,93442,6.8,20.0,93.934676,43.7,36.0,5.6


Priortization Score (Ranked based Composite)

In [9]:
#Top of the list is the higest Priority
# Work on a clean copy
prio = store_summary.reset_index().copy()

# Rank each dimension (higher rank = higher priority)
prio['rank_orders'] = prio['orders'].rank(ascending=True)          # higher volume = higher priority
prio['rank_revenue'] = prio['total_revenue'].rank(ascending=True)
prio['rank_delay'] = prio['avg_readiness_delay'].rank(ascending=True)
prio['rank_on_time'] = prio['pct_on_time_ready'].rank(ascending=False)  # lower on-time = higher priority
prio['rank_fill'] = prio['avg_fill_rate'].rank(ascending=False)
prio['rank_complete'] = prio['pct_complete'].rank(ascending=False)
prio['rank_wait'] = prio['avg_wait'].rank(ascending=True)
prio['rank_abandon'] = prio['abandonment_rate'].rank(ascending=True)

# Composite Priority Score (simple average of ranks)
rank_cols = ['rank_orders', 'rank_revenue', 'rank_delay', 'rank_on_time',
             'rank_fill', 'rank_complete', 'rank_wait', 'rank_abandon']

prio['priority_score'] = prio[rank_cols].mean(axis=1).round(2)

# Final ranked view
priority_ranking = (
    prio[['region', 'store_id', 'orders', 'total_revenue',
          'avg_readiness_delay', 'pct_on_time_ready', 'avg_fill_rate',
          'pct_complete', 'priority_score']]
    .sort_values('priority_score', ascending=False)
    .reset_index(drop=True)
)

priority_ranking

,region,store_id,orders,total_revenue,avg_readiness_delay,pct_on_time_ready,avg_fill_rate,pct_complete,priority_score
0,East,GS-11,731,96870,12.3,15.9,90.109511,22.8,9.69
1,South,GS-09,662,84397,25.2,6.0,84.633838,7.6,8.94
2,South,GS-08,706,97556,9.3,18.8,92.142587,29.2,8.12
3,Central,GS-06,718,98640,10.6,18.4,91.169464,27.6,7.88
4,Central,GS-04,655,83945,19.4,10.2,86.321593,10.7,7.75
5,Central,GS-05,725,99311,8.6,24.6,93.418048,36.1,6.69
6,North,GS-02,713,90806,7.4,22.3,93.775902,39.4,6.12
7,East,GS-10,749,93442,6.8,20.0,93.934676,43.7,6.12
8,West,GS-12,681,87803,8.2,21.3,92.926716,36.1,4.88
9,South,GS-07,723,92215,6.1,19.2,95.014753,49.0,4.25


Adjusted Rank Logic

In [10]:
# Work from the existing prio dataframe (or recreate if needed)
prio = store_summary.reset_index().copy()

# Ranks (higher rank = higher priority)
prio['rank_orders'] = prio['orders'].rank(ascending=True)
prio['rank_revenue'] = prio['total_revenue'].rank(ascending=True)
prio['rank_delay'] = prio['avg_readiness_delay'].rank(ascending=True)
prio['rank_on_time'] = prio['pct_on_time_ready'].rank(ascending=False)
prio['rank_fill'] = prio['avg_fill_rate'].rank(ascending=False)
prio['rank_complete'] = prio['pct_complete'].rank(ascending=False)
prio['rank_wait'] = prio['avg_wait'].rank(ascending=True)
prio['rank_abandon'] = prio['abandonment_rate'].rank(ascending=True)

# Weighted Priority Score
prio['priority_score'] = (
    (prio['rank_delay'] + prio['rank_on_time'] + prio['rank_fill'] + prio['rank_complete']) * 2.0 +
    (prio['rank_orders'] + prio['rank_revenue']) * 1.0 +
    (prio['rank_wait'] + prio['rank_abandon']) * 0.5
).round(2)

# Final ranking
priority_ranking = (
    prio[['region', 'store_id', 'orders', 'total_revenue',
          'avg_readiness_delay', 'pct_on_time_ready', 'avg_fill_rate',
          'pct_complete', 'priority_score']]
    .sort_values('priority_score', ascending=False)
    .reset_index(drop=True)
)

priority_ranking

,region,store_id,orders,total_revenue,avg_readiness_delay,pct_on_time_ready,avg_fill_rate,pct_complete,priority_score
0,South,GS-09,662,84397,25.2,6.0,84.633838,7.6,109.75
1,East,GS-11,731,96870,12.3,15.9,90.109511,22.8,108.75
2,Central,GS-04,655,83945,19.4,10.2,86.321593,10.7,98.00
3,Central,GS-06,718,98640,10.6,18.4,91.169464,27.6,94.50
4,South,GS-08,706,97556,9.3,18.8,92.142587,29.2,88.00
5,Central,GS-05,725,99311,8.6,24.6,93.418048,36.1,71.50
6,East,GS-10,749,93442,6.8,20.0,93.934676,43.7,61.50
7,West,GS-12,681,87803,8.2,21.3,92.926716,36.1,59.75
8,North,GS-02,713,90806,7.4,22.3,93.775902,39.4,58.50
9,South,GS-07,723,92215,6.1,19.2,95.014753,49.0,46.00


Recommendation

**Focus Areas**

GS-09South   Most severe operational pain (25.2 min delay, 6% on-time, 84.6% fill, 7.6% complete)

GS-04Central   Second-highest severity with meaningful volume

GS-11East   Highest volume among underperforming stores + elevated pain

**Recommended Actions (90 days)**

GS-09 and GS-04 (Immediate)
Conduct focused on-hand accuracy audits on high-volume and high-substitution items.
Review substitution patterns and correct inventory/replenishment settings.
Set explicit readiness and fill-rate targets with the store leadership teams.
Hold structured working sessions to surface floor-level obstacles.

GS-11 (Secondary)
Apply the same diagnostic approach at lighter intensity.
Monitor weekly to determine whether the issues are structural or temporary.

Network governance
Track weekly % of orders with readiness delay > 10 minutes and fill rate < 90% at these three stores.
Report progress in the existing operational review cadence.

***Expected Impact***

**Avg readiness delay:** GS-09: 25.2 min / GS-04: 19.4 minMove both below 12 minutes

**On-time readiness:** for GS-09: 6% / GS-04: 10% result: Reach ≥ 18%

**Fill rate:** GS-09: 84.6% / GS-04: 86.3% result:Reach ≥ 92%

**Completeness:** GS-09: 7.6% / GS-04: 10.7% result: Reach ≥ 25%